In [1]:
from nero_interface_server import NeroDualArmServer

In [2]:
import logging
log = logging.getLogger(__name__)

In [3]:
import numpy as np
import time

In [4]:
server = NeroDualArmServer(gripper_enabled=True)

[SERVER] Failed to connect to right arm: CAN socket can_right does not exist.


[Left Arm] 正在获取当前关节角作为 IK 初始基准...
[Left Arm] IK Solver 初始化完成！初始关节角: [-0.641  0.477  0.895  1.657 -0.203  0.24  -0.065]


In [5]:
fp = server.left_robot.get_flange_pose()
robot_pose = np.array(fp.msg)

print("robot z:", robot_pose[2])

robot z: 0.257004


In [6]:
assert server.left_robot is not None, "Left arm failed"
assert server.right_robot is not None, "Right arm failed"
print("Left robot:", server.left_robot)
print("Right robot:", server.right_robot)

Left robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f1cd362dbd0>
Right robot: <pyAgxArm.protocols.can_protocol.drivers.nero.default.driver.Driver object at 0x7f1cd3614dd0>


In [7]:
left_joints = server.left_robot_get_joint_positions()
right_joints = server.right_robot_get_joint_positions()

print("Left joints:", left_joints)
print("Right joints:", right_joints)

Left joints: [-0.6410419809649973, 0.477399910298009, 0.8950048404226922, 1.6567188858705775, -0.20254545969394194, 0.24036674458465906, -0.06538003377970758]
Right joints: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [8]:
left_pose = server.left_robot_get_ee_pose()
right_pose = server.right_robot_get_ee_pose()
print("Left pose:", left_pose)
print("Right pose:", right_pose)

Left pose: [-0.515328, -0.013484999999999999, 0.257004, 1.4787127554596757, -0.3404613771865339, 0.08173376887089445]
Right pose: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [9]:
left_arm_status = server.left_robot_get_arm_status()
right_arm_status = server.right_robot_get_arm_status()

print("Left arm status:", left_arm_status)
print("Right arm status:", right_arm_status)

Left arm status: {'ctrl_mode': CAN_CTRL(0x1), 'arm_status': NORMAL(0x0), 'motion_status': REACH_TARGET_POS_FAILED(0x1), 'trajectory_num': 0}
Right arm status: {'ctrl_mode': 0, 'arm_status': 0, 'motion_status': 0}


In [10]:
server.left_robot_go_home()

left_joints = server.left_robot_get_joint_positions()
print("Left joints:", left_joints)

Left joints: [0.0, -0.12993976281097783, -5.235987755982989e-05, 1.870120293511924, -1.7453292519943296e-05, 5.235987755982989e-05, -0.064943701466709]


In [11]:
# left_joints = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints)

# left_joints_target = np.array(left_joints).copy()
# left_joints_target[0] -= np.deg2rad(0)
# left_joints_target[1] += np.deg2rad(0)
# left_joints_target[2] += np.deg2rad(0)
# left_joints_target[3] += np.deg2rad(0)
# left_joints_target[4] += np.deg2rad(0)
# left_joints_target[5] += np.deg2rad(0)
# left_joints_target[6] += np.deg2rad(0)

# print("Left joints target:", left_joints_target)

# server.left_robot_move_to_joint_positions(left_joints_target, delta=False)

# left_joints_real = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints_real)

In [12]:
left_pose = server.left_robot_get_ee_pose()
print("Left pose:", left_pose)

pose_target = np.array(left_pose).copy()
# end-effector pose [x, y, z, roll, pitch, yaw] (m, rad)
# pose_target[0] += 0.01   # x方向移动1cm
# pose_target[1] += 0.01
# pose_target[2] += 0.01
pose_target = [-0.4, 0.0, 0.4, 1.5708, 0.0, 0.0]

server.left_robot_move_to_ee_pose(pose_target)
time.sleep(3.0)
left_pose = server.left_robot_get_ee_pose()
print("Left pose:", left_pose)

Left pose: [-0.40001, 6.1e-05, 0.38163199999999997, 1.5708137800874167, -0.10442304914682074, -0.00017453292519943296]
Left pose: [-0.399995, 4.9999999999999996e-06, 0.399966, 1.5708661399649764, -5.235987755982989e-05, -6.981317007977319e-05]


In [13]:
# 绝对控制
# target = [0, -70, 0, 150, 0, 0, 30]  # 7维绝对关节角度（度）

# ok = server.servo_j("left_robot", target, delta=False)
# print("servo_j ok:", ok)

In [14]:
# 增量控制
# target_delta = np.zeros(7)
# target_delta[0] = 10

# ok = server.servo_j("left_robot", target_delta.tolist(), delta=True)
# print("servo_j_delta ok:", ok)

In [16]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
print("当前位姿:", current_pose)

# 构造一个小的 delta
delta_pose = np.array([0.02, 0.0, 0.02, 0.0, 0.0, 0.0])
# target_pose = np.array([-0.5, 0.05, 0.2, 1.46, -0.7, -0.04])

# 调用 servo_p（delta 模式）
ret = server.servo_p(
    robot_arm="left_robot",
    pose=delta_pose.tolist(),
    delta=True
    # pose=target_pose.tolist(),
    # delta=False
)

time.sleep(1.5)

fp = server.left_robot.get_flange_pose()
new_pose = np.array(fp.msg, dtype=float)
print("新位姿:", new_pose)

当前位姿: [-3.99995000e-01  1.40000000e-05  3.99966000e-01  1.57086614e+00
 -5.23598776e-05 -8.72664626e-05]
新位姿: [-3.82581000e-01 -2.10000000e-05  4.19582000e-01  1.57104067e+00
  4.80838209e-02  8.72664626e-04]


In [22]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
print("当前位姿:", current_pose)

target_pose = current_pose.copy()
# target_pose[0] += 0.01
# print("目标位姿:", target_pose)

steps = 10
dt = 0.02  # 20Hz

# for i in range(steps):
#     alpha = (i + 1) / steps
#     pose_i = current_pose * (1 - alpha) + target_pose * alpha

#     server.servo_p("left_robot", pose_i.tolist(), delta=False)
#     time.sleep(dt)

for i in range(steps):
    # target_pose[2] += 0.02 / steps   # 每次走一点点
    delta_pose = np.array([-0.02, 0.0, 0.0, 0.0, 0.0, 0.0])
    time1 = time.time()
    server.servo_p("left_robot", delta_pose.tolist(), delta=True)
    time2 = time.time()
    print(f"Step {i+1}/{steps}, servo_p time: {(time2 - time1) * 1000:.2f} ms")
    time.sleep(0.005)

当前位姿: [-0.56916     0.027567    0.109685    1.59071053 -0.72017521 -0.06234316]
Step 1/10, servo_p time: 77.83 ms
Step 2/10, servo_p time: 69.15 ms
Step 3/10, servo_p time: 71.75 ms
Step 4/10, servo_p time: 80.00 ms
Step 5/10, servo_p time: 75.26 ms
Step 6/10, servo_p time: 66.64 ms
Step 7/10, servo_p time: 73.08 ms
Step 8/10, servo_p time: 74.43 ms
Step 9/10, servo_p time: 85.05 ms
Step 10/10, servo_p time: 71.46 ms


In [18]:
# server.left_gripper_goto(0.05)
# # server.left_gripper_grasp()
# print(server.left_gripper_get_state())

In [19]:
# server.stop("left_robot")